DATA READING

In [0]:
df = spark.read.format("parquet")\
    .option("inferSchema", True)\
        .option("header", True)\
             .load("abfss://bronze@carsamdatalake.dfs.core.windows.net/rawdata")

display(df)    

In [0]:
df.printSchema()

DATA TRANSAFORMATION

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import *

In [0]:
# Spliting Model_ID column

df = df.withColumn("ModelCatagory", split(col("Model_ID"), "-")[0])

display(df)

In [0]:
# Carting Unit_Sold from interger to string

df.withColumn("Units_Sold", col("Units_Sold").cast("string")).display()

df.withColumn("Units_Sold", col("Units_Sold").cast("string")).printSchema()

In [0]:
# Calculate Revenue Per Unit

df = df.withColumn("RevPerUnit", col("Revenue")/col("Units_Sold"))
df.display()

AD-HOC

In [0]:
display(df)

In [0]:
# How many unit got sold of each branch every year

df.groupBy("Year", "BranchName")\
    .agg(sum("Units_Sold").alias("TotalUnitSold"))\
        .sort("Year", "TotalUnitSold", ascending=[True, False])\
            .display() 


Databricks visualization. Run in Databricks to view.

DATA WRITING

In [0]:
df.write.format("parquet")\
    .mode("overwrite")\
        .option("path", "abfss://silver@carsamdatalake.dfs.core.windows.net/carsales")\
            .save()

QUERYING SILVER DATA

In [0]:
%sql
select * 
from parquet.`abfss://silver@carsamdatalake.dfs.core.windows.net/carsales`;